# 09 — Pairwise Reward Modeling

## Objective
Explain the logistic pairwise ranking loss, prepare automatically constructed preference pairs, train a scalar scoring model using a separate reward head, and preserve source and checkpoint provenance.

## Research context
A reward model produces scalar values, not a calibrated biomedical truth probability. Reward-model performance should be measured on held-out preference pairs. Avoid describing automatically constructed preference supervision as direct human feedback.

For preferred $y_w$ and rejected $y_l$, the pairwise loss is:

$$\mathcal{L}_{\mathrm{reward}}=-\log\sigma\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right).$$

### A differentiable pairwise example

In [1]:
import torch

score_preferred = torch.tensor(0.6, requires_grad=True)
score_rejected = torch.tensor(0.2, requires_grad=True)
margin = score_preferred - score_rejected
loss = -torch.nn.functional.logsigmoid(margin)
loss.backward()

print("Pairwise loss:", float(loss.item()))
print("Preferred-score gradient:", float(score_preferred.grad.item()))
print("Rejected-score gradient:", float(score_rejected.grad.item()))
assert score_preferred.grad.item() < 0 < score_rejected.grad.item()

Pairwise loss: 0.5130152702331543
Preferred-score gradient: -0.40131232142448425
Rejected-score gradient: 0.40131232142448425


### Output folder and training provenance

In [2]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


### Reward model experiment

The source contains many pair rows per exact prompt. Hold out whole prompt groups before selecting one complete pair per group, so repeated source cases cannot leak between training and validation. The installed TRL RewardTrainer cannot import with this PyTorch build, and the four-bit loading path is incompatible; this notebook uses a bfloat16 scalar sequence classifier with LoRA and an explicit pairwise logistic loss. Both responses must fit the length bound without truncation. The held-out score is specific to synthetic source rankings, not biomedical truth.

The original 25-step run and its `runs/reward_model/` artifact are preserved. Set `RUN_REWARD_ORIGINAL=1` only to reproduce that original run; the follow-up diagnostic and revision use separate controls and output folders.

In [3]:
import os
import json
import time
import random
import hashlib
from collections import defaultdict

if os.getenv("RUN_REWARD_ORIGINAL", "0") == "1":
    import numpy as np
    import torch.nn.functional as F
    from datasets import Dataset, load_dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    from peft import LoraConfig, get_peft_model
    import transformers, peft, datasets

    started = time.perf_counter()
    seed = 42
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    sft_manifest = json.loads((RUNS / "sft_adapter" / "run_manifest.json").read_text(encoding="utf-8"))
    if model_id != sft_manifest["model_id"]:
        raise ValueError("Requested model differs from the cached base model lineage")
    revision = sft_manifest["model_revision"]
    source_manifest = json.loads((RESEARCH / "data" / "med_refl_reasoning_manifest.json").read_text(encoding="utf-8"))
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        cache = Path.home() / ".cache/huggingface/datasets/HANI-LAB___med-refl-dpo/reasoning_enhancement/0.0.0/001e469d8849a7c38a13797b9a52d550ffc36124/med-refl-dpo-train.arrow"
        if hashlib.sha256(cache.read_bytes()).hexdigest() != source_manifest["cache_arrow_sha256"]:
            raise ValueError("Preference cache differs from notebook 07")
        source = Dataset.from_file(str(cache))
        source_location = str(cache)
    else:
        source = load_dataset("HANI-LAB/Med-REFL-DPO", "reasoning_enhancement", split="train")
        source_location = "Hugging Face Datasets load_dataset"
    if len(source) != source_manifest["source_rows"] or not {"instruction", "input", "chosen", "rejected"}.issubset(source.column_names):
        raise ValueError("Preference source schema or size differs from notebook 07")
    groups = defaultdict(list)
    for i, row in enumerate(source):
        prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
        if not prompt or not row["chosen"] or not row["rejected"] or row["chosen"] == row["rejected"]:
            raise ValueError(f"Invalid pair at source row {i}")
        groups[prompt].append(i)
    if len(groups) != source_manifest["unique_prompts"]:
        raise ValueError("Prompt-group count differs from notebook 07")
    group_keys = sorted(groups)
    np.random.default_rng(seed).shuffle(group_keys)
    heldout_groups = set(group_keys[:round(0.2 * len(group_keys))])
    train_groups = [key for key in group_keys if key not in heldout_groups]
    tokenizer = AutoTokenizer.from_pretrained(str(RUNS / "sft_adapter"), local_files_only=True)
    tokenizer.pad_token = tokenizer.eos_token
    max_length = 1024
    def select_pairs(keys, limit):
        selected = []
        screened = 0
        for prompt in keys:
            prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
            indices = list(groups[prompt]); np.random.default_rng(seed + len(prompt)).shuffle(indices)
            for index in indices:
                screened += 1
                row = source[index]
                responses = [prefix + tokenizer(str(row[field]).strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
                             for field in ("chosen", "rejected")]
                if max(map(len, responses)) <= max_length:
                    selected.append({"row": index, "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                                     "chosen_ids": responses[0], "rejected_ids": responses[1]})
                    break
            if len(selected) >= limit:
                break
        return selected, screened
    max_items = int(os.getenv("REWARD_TRAIN_MAX_ITEMS", "100"))
    max_steps = int(os.getenv("REWARD_MAX_STEPS", "25"))
    accumulation = 4
    if not 1 <= max_items <= len(train_groups) or max_steps < 1 or max_steps * accumulation > max_items:
        raise ValueError("Require four distinct training prompt groups per optimizer step")
    training, train_screened = select_pairs(train_groups, max_items)
    heldout, heldout_screened = select_pairs([key for key in group_keys if key in heldout_groups], len(heldout_groups))
    if len(training) < max_items or len(heldout) < 10:
        raise ValueError("Too few complete group-distinct pairs under the token bound")
    assert not {r["prompt_sha256"] for r in training} & {r["prompt_sha256"] for r in heldout}
    print("Source groups:", len(groups), "train groups:", len(train_groups), "held-out groups:", len(heldout_groups))
    print("Selected complete pairs:", len(training), "training and", len(heldout), "held-out")
    print("Maximum selected tokens:", max(len(r[field]) for r in training + heldout for field in ("chosen_ids", "rejected_ids")))
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("This bounded reward-model run requires a CUDA GPU with bfloat16")
    torch.cuda.reset_peak_memory_stats()
    base = AutoModelForSequenceClassification.from_pretrained(model_id, revision=revision,
        local_files_only=True, num_labels=1, dtype=torch.bfloat16).to("cuda")
    base.config.pad_token_id = tokenizer.pad_token_id
    adapter = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"],
                         task_type="SEQ_CLS", modules_to_save=["score"])
    model = get_peft_model(base, adapter).to("cuda")
    model.config.use_cache = False
    def score(ids):
        tokens = torch.tensor([ids], device="cuda")
        return model(input_ids=tokens, use_cache=False).logits.float().reshape(())
    def evaluate(rows):
        model.eval()
        margins = []
        with torch.inference_mode():
            for row in rows:
                margins.append(float((score(row["chosen_ids"]) - score(row["rejected_ids"])).item()))
        return {"n_pairs": len(margins), "pairwise_accuracy": float(np.mean(np.array(margins) > 0)),
                "mean_margin": float(np.mean(margins)), "mean_loss": float(np.mean(np.logaddexp(0, -np.array(margins))))}
    before = evaluate(heldout)
    model.train()
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=5e-5)
    logs = []
    for step in range(max_steps):
        optimizer.zero_grad(set_to_none=True)
        losses, margins = [], []
        for row in training[step * accumulation:(step + 1) * accumulation]:
            margin = score(row["chosen_ids"]) - score(row["rejected_ids"])
            loss = -F.logsigmoid(margin)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite reward pairwise loss")
            (loss / accumulation).backward()
            losses.append(float(loss.detach().item())); margins.append(float(margin.detach().item()))
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        log = {"step": step + 1, "loss": float(np.mean(losses)),
               "pairwise_accuracy": float(np.mean(np.array(margins) > 0)), "mean_margin": float(np.mean(margins))}
        logs.append(log)
        if (step + 1) % 5 == 0:
            print(log, flush=True)
    after = evaluate(heldout)
    output_dir = RUNS / "reward_model"
    output_dir.mkdir(exist_ok=True)
    model.save_pretrained(str(output_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(output_dir))
    manifest = {"variant": "reward_model", "method": "bfloat16 sequence classifier with LoRA and explicit pairwise logistic loss",
        "model_id": model_id, "model_revision": revision, "source": "HANI-LAB/Med-REFL-DPO:reasoning_enhancement",
        "source_location": source_location, "source_fingerprint": source_manifest["source_content_fingerprint"],
        "source_cache_sha256": source_manifest["cache_arrow_sha256"], "seed": seed,
        "split": "seeded exact-prompt-group shuffle; first 20 percent held out before pair selection",
        "train_prompt_groups": len(train_groups), "heldout_prompt_groups": len(heldout_groups),
        "selected_train_rows": [r["row"] for r in training], "selected_heldout_rows": [r["row"] for r in heldout],
        "selected_train_prompt_sha256": [r["prompt_sha256"] for r in training],
        "selected_heldout_prompt_sha256": [r["prompt_sha256"] for r in heldout],
        "train_screened_rows": train_screened, "heldout_screened_rows": heldout_screened,
        "max_length": max_length, "prompt_format": "Qwen user chat template plus assistant prefix; response plus EOS",
        "completed_steps": len(logs), "gradient_accumulation_steps": accumulation, "learning_rate": 5e-5,
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "heldout_before": before, "heldout_after": after, "runtime_seconds": time.perf_counter() - started,
        "peak_gpu_gib": torch.cuda.max_memory_allocated() / 2**30,
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                     "peft": peft.__version__, "datasets": datasets.__version__}, "run_status": "completed"}
    (output_dir / "run_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    (output_dir / "log_history.json").write_text(json.dumps(logs, indent=2) + "\n", encoding="utf-8")
    print("Held-out before:", before, "Held-out after:", after)
    print("Reward model saved:", output_dir)
else:
    print("Original reward-model run disabled. Set RUN_REWARD_ORIGINAL=1 only to reproduce it.")

Original reward-model run disabled. Set RUN_REWARD_ORIGINAL=1 only to reproduce it.


### Training-only fitting diagnostic

A controlled fitting check uses 24 original **training** pairs with different explicit terminal option letters. This syntactic contrast is easy to inspect, but it does not prove the source ranking is clinically correct. Repeated fitting of these same pairs tests whether the score head and adapter can learn pairwise order; it is not a generalization result.

In [4]:
# Training-only memorization diagnostic. This cell is copied into notebook 09.
import os, json, re, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

if os.getenv("RUN_REWARD_DIAGNOSTIC", "0") == "1":
    diagnostic_start = time.perf_counter()
    original_dir = RUNS / "reward_model"
    original = json.loads((original_dir / "run_manifest.json").read_text(encoding="utf-8"))
    source = Dataset.from_file(original["source_location"])
    tokenizer = AutoTokenizer.from_pretrained(str(original_dir), local_files_only=True)
    pattern = re.compile(r"(?i)(?:option|answer(?: is|:)?|choice)\s*[:\-]?\s*([A-E])\b")
    # Distinct explicit terminal options make the pair contrast easy to inspect.
    # This is a training-only diagnostic; the regex does not certify clinical correctness.
    diagnostic_rows = []
    for index in original["selected_train_rows"]:
        row = source[index]
        chosen_option = pattern.findall(row["chosen"][-500:])
        rejected_option = pattern.findall(row["rejected"][-500:])
        if chosen_option and rejected_option and chosen_option[-1].upper() != rejected_option[-1].upper():
            diagnostic_rows.append(index)
        if len(diagnostic_rows) == 24:
            break
    if len(diagnostic_rows) < 16:
        raise ValueError("Insufficient training-only explicit-answer contrasts")
    def diagnostic_ids(row, field):
        prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
        prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
        return prefix + tokenizer(row[field].strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
    diagnostic_pairs = [(i, diagnostic_ids(source[i], "chosen"), diagnostic_ids(source[i], "rejected")) for i in diagnostic_rows]
    if max(len(ids) for _, a, b in diagnostic_pairs for ids in (a, b)) > original["max_length"]:
        raise ValueError("Diagnostic pair exceeds original length bound")
    random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    base = AutoModelForSequenceClassification.from_pretrained(original["model_id"], revision=original["model_revision"],
        local_files_only=True, num_labels=1, dtype=torch.bfloat16).to("cuda")
    base.config.pad_token_id = tokenizer.pad_token_id
    diagnostic_model = PeftModel.from_pretrained(base, str(original_dir), is_trainable=True).to("cuda")
    diagnostic_model.config.use_cache = False
    def diagnostic_score(ids):
        tokens = torch.tensor([ids], device="cuda")
        return diagnostic_model(input_ids=tokens, attention_mask=torch.ones_like(tokens), use_cache=False).logits.float().reshape(())
    def diagnostic_eval():
        diagnostic_model.eval()
        with torch.inference_mode():
            margins = np.array([float((diagnostic_score(a) - diagnostic_score(b)).item()) for _, a, b in diagnostic_pairs])
        return {"accuracy": float(np.mean(margins > 0)), "mean_loss": float(np.mean(np.logaddexp(0, -margins))),
                "margin_quantiles": np.quantile(margins, [0, .25, .5, .75, 1]).tolist()}
    diagnostic_history = [{"step": 0, **diagnostic_eval()}]
    optimizer = torch.optim.AdamW((p for p in diagnostic_model.parameters() if p.requires_grad), lr=1e-4)
    for step in range(1, 61):
        diagnostic_model.train()
        optimizer.zero_grad(set_to_none=True)
        # Cycle through the same 24 training pairs; held-out rows never enter this check.
        for offset in range(4):
            _, chosen_ids, rejected_ids = diagnostic_pairs[((step - 1) * 4 + offset) % len(diagnostic_pairs)]
            margin = diagnostic_score(chosen_ids) - diagnostic_score(rejected_ids)
            loss = -F.logsigmoid(margin)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite diagnostic loss")
            (loss / 4).backward()
        torch.nn.utils.clip_grad_norm_((p for p in diagnostic_model.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        if step % 10 == 0:
            diagnostic_history.append({"step": step, **diagnostic_eval()})
            print("Diagnostic", diagnostic_history[-1], flush=True)
    diagnostic_summary = {"purpose": "training-only fitting check", "source_model": str(original_dir),
        "selected_source_rows": diagnostic_rows, "n_pairs": len(diagnostic_pairs),
        "selection": "first 24 original training pairs with explicit differing terminal option letters",
        "learning_rate": 1e-4, "steps": 60, "pairs_per_step": 4,
        "history": diagnostic_history, "runtime_seconds": time.perf_counter() - diagnostic_start}
    diagnostic_dir = RUNS / "reward_diagnostic"
    diagnostic_dir.mkdir(exist_ok=True)
    (diagnostic_dir / "diagnostic.json").write_text(json.dumps(diagnostic_summary, indent=2) + "\n", encoding="utf-8")
    print("Training-only diagnostic saved:", diagnostic_dir / "diagnostic.json")
else:
    print("Training-only diagnostic disabled.")


Training-only diagnostic disabled.


### Validation-monitored continuation and independent comparison

The original 100 training prompt groups stay in training. Its 71 originally reserved groups stay excluded from development and the new test. The remaining exact prompt groups are split **before** pair selection into 40 development groups, 30 new test groups, and additional training groups. One complete pair is selected per group without inspecting model predictions. Continue the saved 0.5B reward adapter for four epochs, inspect multiple checkpoints on development loss, select once, then score the new test once. The original held-out 61 pairs are reported as an exposed comparison set because their result was already known before this revision. A training-only diagnostic or development score is never presented as independent generalization evidence.

In [5]:
# Follow-up reward experiment. This cell is copied into notebook 09.
import os, json, time, random, hashlib, math
from collections import defaultdict
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from peft import PeftModel

if os.getenv("RUN_REWARD_REVISION", "0") == "1":
    revision_start = time.perf_counter()
    seed = 43  # Fixed partition and pair-selection seed across replications.
    training_seed = int(os.getenv("REWARD_REVISION_SEED", "43"))
    if training_seed not in (43, 44):
        raise ValueError("This audit predeclares only training seeds 43 and 44")
    random.seed(training_seed); np.random.seed(training_seed); torch.manual_seed(training_seed); torch.cuda.manual_seed_all(training_seed)
    original_dir = RUNS / "reward_model"
    original = json.loads((original_dir / "run_manifest.json").read_text(encoding="utf-8"))
    source_manifest = json.loads((RESEARCH / "data" / "med_refl_reasoning_manifest.json").read_text(encoding="utf-8"))
    cache = Path(original["source_location"])
    if hashlib.sha256(cache.read_bytes()).hexdigest() != original["source_cache_sha256"] or original["source_fingerprint"] != source_manifest["source_content_fingerprint"]:
        raise ValueError("Cached preference source differs from the original experiment")
    source = Dataset.from_file(str(cache))
    tokenizer = AutoTokenizer.from_pretrained(str(original_dir), local_files_only=True)
    groups = defaultdict(list)
    for index, row in enumerate(source):
        prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
        groups[prompt].append(index)
    if len(groups) != source_manifest["unique_prompts"]:
        raise ValueError("Preference prompt groups changed")
    original_train_keys = {(str(source[i]["instruction"] or "") + "\n" + str(source[i]["input"] or "")).strip()
                           for i in original["selected_train_rows"]}
    original_holdout_keys = {(str(source[i]["instruction"] or "") + "\n" + str(source[i]["input"] or "")).strip()
                             for i in original["selected_heldout_rows"]}
    # Reconstruct the original 71 reserved groups, including ten without a fitting pair.
    original_group_order = sorted(groups)
    np.random.default_rng(42).shuffle(original_group_order)
    original_reserved = set(original_group_order[:round(.2 * len(original_group_order))])
    if original_holdout_keys - original_reserved or original_train_keys & original_reserved:
        raise ValueError("Original split cannot be reconstructed")
    remaining_keys = sorted(set(groups) - original_train_keys - original_reserved)
    np.random.default_rng(seed).shuffle(remaining_keys)
    development_keys = remaining_keys[:40]
    new_test_keys = remaining_keys[40:70]
    additional_train_keys = remaining_keys[70:]
    assert len(set(development_keys) & set(new_test_keys)) == 0
    assert not (set(development_keys + new_test_keys) & original_train_keys)
    max_length = original["max_length"]
    def encode_pair(index):
        row = source[index]
        prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
        prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
        a = prefix + tokenizer(row["chosen"].strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
        b = prefix + tokenizer(row["rejected"].strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
        return {"row": index, "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
                "chosen_ids": a, "rejected_ids": b, "prompt_tokens": len(prefix)}
    def select_complete(keys, selection_seed):
        chosen, counters = [], {"screened_rows": 0, "overlength_rows": 0, "groups_without_fit": 0}
        for prompt in keys:
            candidates = list(groups[prompt])
            np.random.default_rng(selection_seed + len(prompt)).shuffle(candidates)
            fitted = None
            for index in candidates:
                counters["screened_rows"] += 1
                pair = encode_pair(index)
                if max(len(pair["chosen_ids"]), len(pair["rejected_ids"])) <= max_length:
                    fitted = pair
                    break
                counters["overlength_rows"] += 1
            if fitted is None:
                counters["groups_without_fit"] += 1
            else:
                chosen.append(fitted)
        return chosen, counters
    original_train = [encode_pair(i) for i in original["selected_train_rows"]]
    additional_train, additional_counts = select_complete(additional_train_keys, seed)
    development, development_counts = select_complete(development_keys, seed)
    new_test, test_counts = select_complete(new_test_keys, seed)
    original_holdout = [encode_pair(i) for i in original["selected_heldout_rows"]]
    training = original_train + additional_train
    if min(len(training), len(development), len(new_test)) < 20:
        raise ValueError("Too few complete pairs for the planned split")
    hashes = [{p["prompt_sha256"] for p in rows} for rows in (training, development, new_test, original_holdout)]
    if any(hashes[i] & hashes[j] for i in range(4) for j in range(i + 1, 4)):
        raise ValueError("Prompt-group leakage across partitions")
    print("Complete pairs train/development/new test/original holdout:",
          len(training), len(development), len(new_test), len(original_holdout))
    print("Screening counts:", additional_counts, development_counts, test_counts)

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("A bfloat16 CUDA GPU is required")
    torch.cuda.reset_peak_memory_stats()
    def load_reward(trainable):
        base = AutoModelForSequenceClassification.from_pretrained(original["model_id"], revision=original["model_revision"],
            local_files_only=True, num_labels=1, dtype=torch.bfloat16).to("cuda")
        base.config.pad_token_id = tokenizer.pad_token_id
        model = PeftModel.from_pretrained(base, str(original_dir), is_trainable=trainable).to("cuda")
        model.config.use_cache = False
        return model
    model = load_reward(True)
    def score(ids):
        tokens = torch.tensor([ids], device="cuda")
        return model(input_ids=tokens, attention_mask=torch.ones_like(tokens), use_cache=False).logits.float().reshape(())
    def score_rows(rows):
        model.eval()
        scores = []
        with torch.inference_mode():
            for pair in rows:
                chosen = float(score(pair["chosen_ids"]).item())
                rejected = float(score(pair["rejected_ids"]).item())
                scores.append((chosen, rejected))
        return np.asarray(scores)
    def summarize(scores):
        margins = scores[:, 0] - scores[:, 1]
        return {"n_pairs": int(len(margins)), "accuracy": float(np.mean(margins > 0)),
                "mean_loss": float(np.mean(np.logaddexp(0, -margins))),
                "mean_margin": float(np.mean(margins)),
                "chosen_score_quantiles": np.quantile(scores[:, 0], [0, .1, .25, .5, .75, .9, 1]).tolist(),
                "rejected_score_quantiles": np.quantile(scores[:, 1], [0, .1, .25, .5, .75, .9, 1]).tolist(),
                "margin_quantiles": np.quantile(margins, [0, .1, .25, .5, .75, .9, 1]).tolist()}
    original_train_scores = score_rows(training)
    original_dev_scores = score_rows(development)
    checkpoints = [{"step": 0, "train": summarize(original_train_scores), "development": summarize(original_dev_scores)}]
    revision_dir = RUNS / ("reward_model_revision" if training_seed == 43 else "reward_model_revision_seed44")
    revision_dir.mkdir(exist_ok=True)
    # Exactly four epochs over complete, group-distinct training pairs.
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-5)
    order = []
    for epoch in range(4):
        order.extend(np.random.default_rng(training_seed + epoch).permutation(len(training)).tolist())
    accumulation = 4
    steps = len(order) // accumulation
    checkpoint_every = max(1, steps // 4)
    step_logs = []
    for step in range(1, steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        losses = []
        for index in order[(step - 1) * accumulation:step * accumulation]:
            pair = training[index]
            margin = score(pair["chosen_ids"]) - score(pair["rejected_ids"])
            loss = -F.logsigmoid(margin)
            if not torch.isfinite(loss):
                raise FloatingPointError("Nonfinite revision loss")
            (loss / accumulation).backward()
            losses.append(float(loss.detach().item()))
        torch.nn.utils.clip_grad_norm_((p for p in model.parameters() if p.requires_grad), 1.0)
        optimizer.step()
        step_logs.append({"step": step, "batch_mean_loss": float(np.mean(losses))})
        if step % checkpoint_every == 0 or step == steps:
            train_scores = score_rows(training)
            dev_scores = score_rows(development)
            report = {"step": step, "train": summarize(train_scores), "development": summarize(dev_scores)}
            checkpoints.append(report)
            checkpoint_dir = revision_dir / f"checkpoint_{step:03d}"
            model.save_pretrained(str(checkpoint_dir), safe_serialization=True)
            print("Checkpoint", step, "train", report["train"]["accuracy"],
                  "development", report["development"]["accuracy"],
                  "development loss", report["development"]["mean_loss"], flush=True)
    # Select by development loss only. The new test and original holdout are scored after this choice.
    selected = min(checkpoints, key=lambda r: (r["development"]["mean_loss"], r["step"]))
    selected_step = selected["step"]
    if selected_step == 0:
        selected_dir = original_dir
    else:
        selected_dir = revision_dir / f"checkpoint_{selected_step:03d}"
    del model
    torch.cuda.empty_cache()
    base = AutoModelForSequenceClassification.from_pretrained(original["model_id"], revision=original["model_revision"],
        local_files_only=True, num_labels=1, dtype=torch.bfloat16).to("cuda")
    base.config.pad_token_id = tokenizer.pad_token_id
    model = PeftModel.from_pretrained(base, str(selected_dir), is_trainable=False).to("cuda")
    model.config.use_cache = False
    selected_test_scores = score_rows(new_test)
    selected_old_holdout_scores = score_rows(original_holdout)
    selected_train_scores = score_rows(training)
    # Reuse the already measured starting checkpoint only for train/development.
    # Score the starting checkpoint on the new test once, after checkpoint selection.
    if selected_step == 0:
        starting_test_scores = selected_test_scores.copy()
    else:
        del model
        torch.cuda.empty_cache()
        model = load_reward(False)
        starting_test_scores = score_rows(new_test)
    start_correct = (starting_test_scores[:, 0] - starting_test_scores[:, 1]) > 0
    selected_correct = (selected_test_scores[:, 0] - selected_test_scores[:, 1]) > 0
    paired_difference = selected_correct.astype(int) - start_correct.astype(int)
    boot = np.random.default_rng(20261020).integers(0, len(new_test), size=(20000, len(new_test)))
    paired_interval = np.quantile(paired_difference[boot].mean(axis=1), [.025, .975]).tolist()
    def wilson(successes, n):
        z = 1.95996398454
        p = successes / n
        center = (p + z*z/(2*n))/(1+z*z/n)
        radius = z*math.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1+z*z/n)
        return [center-radius, center+radius]
    test_cases = []
    for pair, start_pair, revised_pair in zip(new_test, starting_test_scores, selected_test_scores):
        test_cases.append({"source_row": pair["row"], "prompt_sha256": pair["prompt_sha256"],
                           "starting_scores": start_pair.tolist(), "selected_scores": revised_pair.tolist(),
                           "starting_margin": float(start_pair[0]-start_pair[1]),
                           "selected_margin": float(revised_pair[0]-revised_pair[1])})
    summary = {"variant": "reward_model_revision", "method": "continued original 0.5B LoRA reward adapter",
        "starting_checkpoint": str(original_dir), "selected_checkpoint": str(selected_dir),
        "model_id": original["model_id"], "model_revision": original["model_revision"],
        "source_fingerprint": original["source_fingerprint"], "source_cache_sha256": original["source_cache_sha256"],
        "split_seed": seed, "selection_seed": seed, "training_seed": training_seed, "max_length": max_length,
        "partition_rule": "original 100 train groups retained; original 71 reserved groups excluded; remaining groups shuffled into 40 development, 30 new test, rest additional train",
        "counts": {"train_pairs": len(training), "development_pairs": len(development), "new_test_pairs": len(new_test),
                   "original_holdout_pairs": len(original_holdout), "additional_screening": additional_counts,
                   "development_screening": development_counts, "new_test_screening": test_counts},
        "partition_rows": {"train": [p["row"] for p in training], "development": [p["row"] for p in development],
                           "new_test": [p["row"] for p in new_test], "original_holdout": [p["row"] for p in original_holdout]},
        "partition_prompt_sha256": {"train": [p["prompt_sha256"] for p in training],
                                    "development": [p["prompt_sha256"] for p in development],
                                    "new_test": [p["prompt_sha256"] for p in new_test],
                                    "original_holdout": [p["prompt_sha256"] for p in original_holdout]},
        "epochs": 4, "steps": steps, "gradient_accumulation_steps": accumulation,
        "learning_rate": 3e-5, "checkpoint_rule": "minimum development pairwise loss; earliest step on tie",
        "selected_step": selected_step, "checkpoints": checkpoints,
        "selected_train": summarize(selected_train_scores),
        "starting_new_test": summarize(starting_test_scores), "selected_new_test": summarize(selected_test_scores),
        "selected_original_holdout": summarize(selected_old_holdout_scores),
        "new_test_starting_wilson95": wilson(int(start_correct.sum()), len(new_test)),
        "new_test_selected_wilson95": wilson(int(selected_correct.sum()), len(new_test)),
        "new_test_paired_difference_bootstrap95": paired_interval,
        "new_test_case_scores": test_cases, "runtime_seconds": time.perf_counter() - revision_start,
        "peak_gpu_gib": torch.cuda.max_memory_allocated() / 2**30, "run_status": "completed"}
    (revision_dir / "run_manifest.json").write_text(json.dumps(summary, indent=2) + "\n", encoding="utf-8")
    (revision_dir / "step_logs.json").write_text(json.dumps(step_logs, indent=2) + "\n", encoding="utf-8")
    tokenizer.save_pretrained(str(revision_dir))
    print("Selected checkpoint step:", selected_step)
    print("New test starting:", summary["starting_new_test"])
    print("New test selected:", summary["selected_new_test"])
    print("Original holdout selected:", summary["selected_original_holdout"])
    print("Paired new-test difference interval:", paired_interval)
else:
    print("Reward-model revision disabled.")


Reward-model revision disabled.


### Read-only audit and checkpoint verification

The following cells summarize the preserved original run, training-only diagnostic, and two predeclared continuation seeds. They recompute source length and ambiguity checks and reload the selected checkpoints. The new test pairs were scored once per fixed run after development-based checkpoint selection. Accuracy refers to the source ranking, not verified clinical truth.

In [6]:
# Read-only report from the preserved original, diagnostic, and revision artifacts.
import json, re, hashlib
from pathlib import Path
import numpy as np
from datasets import Dataset
from transformers import AutoTokenizer

original = json.loads((RUNS / "reward_model" / "run_manifest.json").read_text(encoding="utf-8"))
diagnostic = json.loads((RUNS / "reward_diagnostic" / "diagnostic.json").read_text(encoding="utf-8"))
revisions = [json.loads((RUNS / name / "run_manifest.json").read_text(encoding="utf-8"))
             for name in ("reward_model_revision", "reward_model_revision_seed44")]
source_manifest = json.loads((RESEARCH / "data" / "med_refl_reasoning_manifest.json").read_text(encoding="utf-8"))
source_path = Path(original["source_location"])
assert hashlib.sha256(source_path.read_bytes()).hexdigest() == source_manifest["cache_arrow_sha256"]
source = Dataset.from_file(str(source_path))
tokenizer = AutoTokenizer.from_pretrained(str(RUNS / "reward_model"), local_files_only=True)
selected_rows = original["selected_train_rows"] + original["selected_heldout_rows"]
length_rows = []
for index in selected_rows:
    row = source[index]
    prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
    prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
    chosen_length = len(prefix) + len(tokenizer(row["chosen"].strip(), add_special_tokens=False).input_ids) + 1
    rejected_length = len(prefix) + len(tokenizer(row["rejected"].strip(), add_special_tokens=False).input_ids) + 1
    length_rows.append((chosen_length, rejected_length))
lengths = np.asarray(length_rows)
option_pattern = re.compile(r"(?i)(?:option|answer(?: is|:)?|choice)\s*[:\-]?\s*([A-E])\b")
same_option = differing_option = unknown_option = 0
for index in selected_rows:
    row = source[index]
    a = option_pattern.findall(row["chosen"][-500:]); b = option_pattern.findall(row["rejected"][-500:])
    if not a or not b:
        unknown_option += 1
    elif a[-1].upper() == b[-1].upper():
        same_option += 1
    else:
        differing_option += 1
print("SOURCE", source_manifest["source_rows"], "rows;", source_manifest["unique_prompts"],
      "exact prompt groups; rejected invalid rows", sum(source_manifest["rejected_records"].values()))
print("ORIGINAL_SELECTION 100 train + 61 held-out pairs; 150 + 407 screened overlength rows; 10 reserved groups without a fitting pair")
print("SELECTED_TOKEN_LENGTHS chosen", np.quantile(lengths[:, 0], [0,.25,.5,.75,1]).tolist(),
      "rejected", np.quantile(lengths[:, 1], [0,.25,.5,.75,1]).tolist())
print("LENGTH_DIRECTION chosen longer", int(np.sum(lengths[:,0] > lengths[:,1])),
      "rejected longer", int(np.sum(lengths[:,0] < lengths[:,1])),
      "equal", int(np.sum(lengths[:,0] == lengths[:,1])))
print("TERMINAL_OPTION_AUDIT same", same_option, "different", differing_option,
      "unparseable", unknown_option, "(regex is descriptive, not a truth label)")
print("DIAGNOSTIC", diagnostic["n_pairs"], "training-only pairs; accuracy at step 0 and 60:",
      diagnostic["history"][0]["accuracy"], diagnostic["history"][-1]["accuracy"])
print("ORIGINAL held-out before/after", original["heldout_before"]["pairwise_accuracy"],
      original["heldout_after"]["pairwise_accuracy"])
for result in revisions:
    chosen_checkpoint = next(c for c in result["checkpoints"] if c["step"] == result["selected_step"])
    print("REVISION seed", result["training_seed"] if "training_seed" in result else 43,
          "selected step", result["selected_step"],
          "train", result["selected_train"]["accuracy"],
          "development", chosen_checkpoint["development"]["accuracy"],
          "new test", result["selected_new_test"]["accuracy"],
          "new test loss", result["selected_new_test"]["mean_loss"])
first = revisions[0]
cases = first["new_test_case_scores"]
for label, case in (("source-ranked failure", min(cases, key=lambda c: c["selected_margin"])),
                    ("source-ranked success", max(cases, key=lambda c: c["selected_margin"]))):
    row = source[case["source_row"]]
    print("CASE", label, "source row", case["source_row"], "margin", case["selected_margin"],
          "prompt excerpt", repr(row["instruction"][:110]),
          "chosen ending", repr(row["chosen"][-150:]),
          "rejected ending", repr(row["rejected"][-150:]))
ambiguous = source[19325]
print("AMBIGUOUS_PAIR source row 19325: both responses end at the same 90% answer; source ranking may reflect subtle reasoning or style differences")


SOURCE 21326 rows; 355 exact prompt groups; rejected invalid rows 0
ORIGINAL_SELECTION 100 train + 61 held-out pairs; 150 + 407 screened overlength rows; 10 reserved groups without a fitting pair
SELECTED_TOKEN_LENGTHS chosen [544.0, 748.0, 822.0, 894.0, 1024.0] rejected [562.0, 748.0, 832.0, 932.0, 1022.0]
LENGTH_DIRECTION chosen longer 66 rejected longer 92 equal 3
TERMINAL_OPTION_AUDIT same 17 different 48 unparseable 96 (regex is descriptive, not a truth label)
DIAGNOSTIC 24 training-only pairs; accuracy at step 0 and 60: 0.625 1.0
ORIGINAL held-out before/after 0.47540983606557374 0.5081967213114754
REVISION seed 43 selected step 45 train 0.6010928961748634 development 0.5294117647058824 new test 0.4090909090909091 new test loss 1.007447817261363
REVISION seed 44 selected step 45 train 0.5792349726775956 development 0.5588235294117647 new test 0.4090909090909091 new test loss 1.0257618246884794
CASE source-ranked failure source row 20443 margin -2.859375 prompt excerpt 'A 68-year-

### Immediate interpretation of the reward audit

These printed scores come from preserved original and revised artifacts; the training cells above are disabled in the latest execution. The newly reserved 22-pair preference test did not improve with either development-selected continuation (9/22 versus 11/22 for the starting adapter). Source-ranked success and failure excerpts are agreement with automatic labels, not clinical adjudication.

In [7]:
# Fresh checkpoint reload: stored scores must be reproduced on the same complete test inputs.
import torch
from transformers import AutoModelForSequenceClassification
from peft import PeftModel

def verify_selected_scores(result):
    model_base = AutoModelForSequenceClassification.from_pretrained(
        result["model_id"], revision=result["model_revision"], local_files_only=True,
        num_labels=1, dtype=torch.bfloat16).to("cuda")
    model_base.config.pad_token_id = tokenizer.pad_token_id
    restored = PeftModel.from_pretrained(model_base, result["selected_checkpoint"], is_trainable=False).to("cuda").eval()
    differences = []
    with torch.inference_mode():
        for case in result["new_test_case_scores"]:
            row = source[case["source_row"]]
            prompt = (str(row["instruction"] or "") + "\n" + str(row["input"] or "")).strip()
            prefix = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=True, add_generation_prompt=True).input_ids
            measured = []
            for field in ("chosen", "rejected"):
                ids = prefix + tokenizer(row[field].strip(), add_special_tokens=False).input_ids + [tokenizer.eos_token_id]
                tokens = torch.tensor([ids], device="cuda")
                measured.append(float(restored(input_ids=tokens, attention_mask=torch.ones_like(tokens), use_cache=False).logits.float().reshape(()).item()))
            differences.extend(abs(a-b) for a,b in zip(measured, case["selected_scores"]))
    del restored, model_base
    torch.cuda.empty_cache()
    return max(differences)

reload_differences = [verify_selected_scores(result) for result in revisions]
print("Maximum absolute score difference after fresh reload, seeds 43 and 44:", reload_differences)
assert all(value == 0 for value in reload_differences)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B-Instruct
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Maximum absolute score difference after fresh reload, seeds 43 and 44: [0.0, 0.0]


### Results

The latest execution audited preserved runs and verified reload without retraining. The original 25-step adapter moved from 29/61 to 31/61 on an exposed holdout. A 24-pair training-only diagnostic fit all 24. Two development-selected continuations each scored 9/22 on a newly reserved group-disjoint preference test, versus 11/22 for the starting reward adapter. Reload reproduced stored scores exactly.

### Interpretation

The scalar head and adapter can fit clear training contrasts, but selected checkpoints did not improve independent preference discrimination.

### Limitations

The new test is small, the exposed holdout cannot support fresh selection, and automated preference labels may be ambiguous or biased by style and length. Reload integrity does not validate ranking quality.

### Key Observations

The reward model is not a validated Reinforcement Learning from Human Feedback (RLHF) or Proximal Policy Optimization (PPO) signal; notebook 10 does not use it.


## Key observations

The score head and Parameter-Efficient Fine-Tuning (PEFT) adapter can memorize 24 clear-contrast training pairs, but the full-source preference ranking did not generalize in the independent group-disjoint test. The original and both longer runs remain unsuitable as reward signals for reinforcement learning from human feedback (RLHF) or Proximal Policy Optimization (PPO). The labels are automatically constructed rather than verified human judgments, and some pairs share the same terminal answer. The new test has only 22 length-eligible groups; exact prompt separation does not exclude semantic near-duplicates. Future work should audit ranking validity with domain experts, construct a larger independent group-disjoint evaluation set, and compare a stronger model as a separate experiment only if justified. No downstream notebook is run here.